In [ ]:
# ============================================================
# VALIDATION A — BRANCH A
# D11 — Siyaram Silk Mills Limited
#       Investor Presentation Q4 & FY24
# ============================================================
#
# Methodology stage covered:
# Stage 4 — Post-processing and Validation
#
# Compares the preserved Branch A extraction against the
# fixed Stage 1 document-grounded reference dataset.
# ============================================================

from google.colab import files
from pathlib import Path
from collections import Counter, defaultdict
from difflib import SequenceMatcher

import hashlib
import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment


In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D11"

DOCUMENT_NAME = (
    "Siyaram Silk Mills Limited — Investor Presentation Q4 & FY24"
)

BRANCH = "A"

BRANCH_NAME = "Direct Ingestion"

INPUT_REPRESENTATION = "Original investor-presentation PDF"

EXPECTED_SOURCE_SHA256 = (
    "604c536562921441733fa3a2b95d3cd43da9aa9f9a665ced87e59c88c5bdd952"
)

EXPECTED_RECORD_COUNT = 199

EXPECTED_CATEGORY_COUNTS = {
    "Presentation metadata": 3,
    "Management commentary": 17,
    "Quarterly business performance": 45,
    "Profit and loss statement": 102,
    "Company profile": 8,
    "Corporate timeline": 17,
    "Operational footprint": 7,
}

FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location",
]

MANDATORY_STRING_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Reporting Period",
    "Source Location",
]

VALUE_ALLOWED_TYPES = (
    str,
    int,
    float,
    type(None),
)


ALIGNMENT_IDENTITY_FIELDS = [
    "Category",
    "Topic",
    "Reporting Period",
]

PRIMARY_CORRECTNESS_FIELDS = [
    "Value",
    "Unit",
    "Source Location",
]

DIAGNOSTIC_FIELDS = [
    "Description",
]

EXPECTED_SOURCE_PAGES = {
    1, 2, 4, 5, 6, 8, 9, 10
}

EXPECTED_QUALIFIED_VALUES = {
    "800+",
    "~100",
    "245+",
    "~1.85",
    "~4.5",
    "5 and counting",
}

OUTPUT_DIR = Path(
    "outputs_D11_validation_A_revised"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Document:", DOCUMENT_ID)
print("Expected records:", EXPECTED_RECORD_COUNT)
print(
    "Alignment identity fields:",
    ALIGNMENT_IDENTITY_FIELDS
)
print("Primary correctness fields:", PRIMARY_CORRECTNESS_FIELDS)
print("Output directory:", OUTPUT_DIR)


In [ ]:
# ============================================================
# 2. Upload canonical validation inputs
# ============================================================

print(
    "Upload exactly four files:\n"
    "1. D11_reference_values.csv\n"
    "2. D11_branch_A_parsed_extraction.json\n"
    "3. D11_branch_A_technical_diagnostics.json\n"
    "4. D11_branch_A_experiment_metadata.json"
)

uploaded = files.upload()

required_names = {
    "D11_reference_values.csv",
    "D11_branch_A_parsed_extraction.json",
    "D11_branch_A_technical_diagnostics.json",
    "D11_branch_A_experiment_metadata.json",
}

observed_names = set(uploaded.keys())

if observed_names != required_names:
    raise ValueError(
        "Upload exactly the four canonical files listed above. "
        f"Observed: {sorted(observed_names)}"
    )

REFERENCE_PATH = Path("D11_reference_values.csv")
EXTRACTION_PATH = Path("D11_branch_A_parsed_extraction.json")
TECHNICAL_DIAGNOSTICS_PATH = Path("D11_branch_A_technical_diagnostics.json")
METADATA_PATH = Path("D11_branch_A_experiment_metadata.json")


In [ ]:
# ============================================================
# 3. Hashing and load inputs
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


REFERENCE_SHA256 = sha256_file(
    REFERENCE_PATH
)

EXTRACTION_SHA256 = sha256_file(
    EXTRACTION_PATH
)

TECHNICAL_DIAGNOSTICS_SHA256 = sha256_file(
    TECHNICAL_DIAGNOSTICS_PATH
)

METADATA_SHA256 = sha256_file(
    METADATA_PATH
)


# ------------------------------------------------------------
# Fixed Stage 1 reference
# ------------------------------------------------------------

reference_df = pd.read_csv(
    REFERENCE_PATH,
    dtype=object,
    keep_default_na=True,
)

reference_df = reference_df.where(
    pd.notna(reference_df),
    None,
)


def restore_reference_value(value):
    if value is None or pd.isna(value):
        return None

    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return value

    text = str(value).strip()

    if (
        text.startswith("~")
        or text.endswith("+")
        or "and counting" in text.casefold()
    ):
        return text

    if re.fullmatch(
        r"-?\d+(?:\.\d+)?",
        text,
    ):
        number = float(text)

        return (
            int(number)
            if number.is_integer()
            else number
        )

    return text


reference_df["Value"] = (
    reference_df["Value"]
    .map(restore_reference_value)
)


# ------------------------------------------------------------
# Canonical Branch A extraction
# ------------------------------------------------------------

parsed_extraction = json.loads(
    EXTRACTION_PATH.read_text(
        encoding="utf-8"
    )
)

top_level_object_valid = isinstance(
    parsed_extraction,
    dict,
)

document_id_correct = (
    top_level_object_valid
    and parsed_extraction.get("document_id")
    == DOCUMENT_ID
)

branch_correct = (
    top_level_object_valid
    and parsed_extraction.get("branch")
    == BRANCH
)

records_is_list = (
    top_level_object_valid
    and isinstance(
        parsed_extraction.get("records"),
        list,
    )
)

if not records_is_list:
    raise ValueError(
        "Canonical parsed extraction must contain a records list."
    )

extracted_records = parsed_extraction[
    "records"
]

extracted_df = pd.DataFrame(
    extracted_records
)


# ------------------------------------------------------------
# Branch A technical/provenance artifacts
# ------------------------------------------------------------

technical_diagnostics = json.loads(
    TECHNICAL_DIAGNOSTICS_PATH.read_text(
        encoding="utf-8"
    )
)

experiment_metadata = json.loads(
    METADATA_PATH.read_text(
        encoding="utf-8"
    )
)

branch_a_structurally_evaluable = bool(
    technical_diagnostics.get(
        "structurally_evaluable",
        False
    )
)

parsed_extraction_hash_matches_metadata = (
    experiment_metadata.get(
        "parsed_extraction_sha256"
    )
    == EXTRACTION_SHA256
)

source_hash_matches_stage_1 = (
    experiment_metadata.get(
        "source_sha256"
    )
    == EXPECTED_SOURCE_SHA256
)

print("Reference SHA-256:", REFERENCE_SHA256)
print("Extraction SHA-256:", EXTRACTION_SHA256)
print("Branch A structurally evaluable:", branch_a_structurally_evaluable)
print(
    "Parsed extraction hash matches metadata:",
    parsed_extraction_hash_matches_metadata,
)
print(
    "Source hash matches Stage 1:",
    source_hash_matches_stage_1,
)


In [ ]:
# ============================================================
# 4. Schema, types and content diagnostics
# ============================================================

reference_schema_exact = (
    reference_df.columns.tolist()
    == FIELDS
)

extraction_schema_exact = (
    extracted_df.columns.tolist()
    == FIELDS
)


def validate_types(records, dataset_name):
    issues = []

    for index, record in enumerate(records):

        if not isinstance(record, dict):
            issues.append({
                "dataset": dataset_name,
                "record_index": index,
                "field": None,
                "issue": "Record is not an object",
            })
            continue

        if set(record.keys()) != set(FIELDS):
            issues.append({
                "dataset": dataset_name,
                "record_index": index,
                "field": None,
                "issue": "Field set differs from schema",
                "observed_fields": list(record.keys()),
            })

        for field in MANDATORY_STRING_FIELDS:
            value = record.get(field)

            if value is None or (
                isinstance(value, str)
                and not value.strip()
            ):
                issues.append({
                    "dataset": dataset_name,
                    "record_index": index,
                    "field": field,
                    "issue": "Missing mandatory value",
                })

            elif not isinstance(value, str):
                issues.append({
                    "dataset": dataset_name,
                    "record_index": index,
                    "field": field,
                    "issue": "Expected string",
                    "observed_type": type(value).__name__,
                })

        value = record.get("Value")

        if (
            isinstance(value, bool)
            or not isinstance(
                value,
                VALUE_ALLOWED_TYPES,
            )
        ):
            issues.append({
                "dataset": dataset_name,
                "record_index": index,
                "field": "Value",
                "issue": "Expected string, number or null",
                "observed_type": type(value).__name__,
            })

    return issues


reference_type_issues = validate_types(
    reference_df.to_dict("records"),
    "Reference",
)

extraction_type_issues = validate_types(
    extracted_records,
    "Extraction",
)

reference_types_valid = (
    len(reference_type_issues) == 0
)

extraction_types_valid = (
    len(extraction_type_issues) == 0
)

reference_record_count_valid = (
    len(reference_df)
    == EXPECTED_RECORD_COUNT
)

extraction_record_count_valid = (
    len(extracted_df)
    == EXPECTED_RECORD_COUNT
)

reference_category_counts = (
    reference_df["Category"]
    .value_counts()
    .to_dict()
)

extraction_category_counts = (
    extracted_df["Category"]
    .value_counts()
    .to_dict()
)

reference_category_counts_valid = (
    reference_category_counts
    == EXPECTED_CATEGORY_COUNTS
)

extraction_category_counts_valid = (
    extraction_category_counts
    == EXPECTED_CATEGORY_COUNTS
)
schema_validity = bool(
    branch_a_structurally_evaluable
)

schema_diagnostics = {
    "top_level_object_valid":
        bool(top_level_object_valid),

    "document_id_correct":
        bool(document_id_correct),

    "branch_correct":
        bool(branch_correct),

    "records_is_list":
        bool(records_is_list),

    "local_record_schema_valid":
        bool(extraction_schema_exact),

    "local_field_types_valid":
        bool(extraction_types_valid),

    "structurally_evaluable":
        bool(branch_a_structurally_evaluable),
}

print("Reference schema exact:", reference_schema_exact)
print("Extraction schema exact:", extraction_schema_exact)
print("Reference types valid:", reference_types_valid)
print("Extraction types valid:", extraction_types_valid)
print("Schema validity:", schema_validity)
print("Reference count:", len(reference_df))
print("Extraction count:", len(extracted_df))
print("Reference category counts:", reference_category_counts)
print("Extraction category counts:", extraction_category_counts)


In [ ]:
# ============================================================
# 5. Confirmation of current Stage 1 D11 reference semantics
# ============================================================

def extract_page(value):
    if value is None:
        return None

    match = re.fullmatch(
        r"PDF page\s+(\d+)",
        str(value).strip(),
        flags=re.IGNORECASE,
    )

    if not match:
        return None

    return int(
        match.group(1)
    )


observed_source_pages = {
    page
    for page in (
        reference_df["Source Location"]
        .map(extract_page)
    )
    if page is not None
}

observed_qualified_values = {
    str(value)
    for value in reference_df["Value"]
    if str(value) in EXPECTED_QUALIFIED_VALUES
}

identity_duplicate_count = int(
    reference_df.duplicated(
        subset=ALIGNMENT_IDENTITY_FIELDS,
        keep=False,
    ).sum()
)

timeline_year_units_valid = all([
    reference_df.loc[
        (reference_df["Category"] == "Corporate timeline")
        & (reference_df["Topic"] == "Established"),
        "Unit",
    ].eq("year").all(),

    reference_df.loc[
        (reference_df["Category"] == "Corporate timeline")
        & (reference_df["Topic"] == "Public listing"),
        "Unit",
    ].eq("year").all(),
])

reference_semantic_checks = {
    "reference_schema_exact":
        bool(reference_schema_exact),

    "reference_types_valid":
        bool(reference_types_valid),

    "reference_record_count_valid":
        bool(reference_record_count_valid),

    "reference_category_counts_valid":
        bool(reference_category_counts_valid),

    "reference_identity_unique":
        identity_duplicate_count == 0,

    "source_page_coverage_valid":
        observed_source_pages
        == EXPECTED_SOURCE_PAGES,

    "qualified_values_preserved":
        observed_qualified_values
        == EXPECTED_QUALIFIED_VALUES,

    "timeline_year_units_valid":
        bool(
            timeline_year_units_valid
        ),

    "no_reference_records_from_divider_pages":
        not bool(
            observed_source_pages
            & {3, 7}
        ),
}

reference_semantics_valid = all(
    reference_semantic_checks.values()
)

print(
    json.dumps(
        reference_semantic_checks,
        ensure_ascii=False,
        indent=2,
    )
)

print(
    "Current D11 reference semantics valid:",
    reference_semantics_valid,
)

if not reference_semantics_valid:
    raise AssertionError(
        "The supplied D11 reference does not match the "
        "current frozen Stage 1 D11 reference semantics."
    )


In [ ]:
# ============================================================
# 6. Comparison-only normalisation
# ============================================================

def normalise_text(value):
    if value is None:
        return ""

    text = unicodedata.normalize(
        "NFKC",
        str(value),
    )

    text = (
        text
        .replace("\u00a0", " ")
        .replace("\u2007", " ")
        .replace("\u202f", " ")
        .replace("—", "-")
        .replace("–", "-")
        .replace("‑", "-")
        .replace("’", "'")
        .replace("“", '"')
        .replace("”", '"')
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text.casefold()


def normalise_topic(value):
    return normalise_text(
        value
    )


PERIOD_EQUIVALENCE = {
    "q4fy24": "q4 fy24",
    "q4fy23": "q4 fy23",
    "q3fy24": "q3 fy24",
    "q4 & fy24": "q4 & fy24",
    "q4 and fy24": "q4 & fy24",
    "31 march 2024": "2024-03-31",
    "march 31, 2024": "2024-03-31",
    "march 31 2024": "2024-03-31",
    "2024-03-31": "2024-03-31",
}


def canonical_period(value):
    text = normalise_text(
        value
    )

    return PERIOD_EQUIVALENCE.get(
        text,
        text,
    )


UNIT_EQUIVALENCE = {
    "text": "text",
    "rs. mn": "rs mn",
    "rs mn": "rs mn",
    "₹ mn": "rs mn",
    "₹ in mn": "rs mn",
    "rs. crores": "rs crores",
    "rs crores": "rs crores",
    "rs. per share": "rs per share",
    "rs per share": "rs per share",
    "rs.": "rs",
    "rs": "rs",
    "%": "percent",
    "percentage": "percent",
    "percent": "percent",
    "store": "stores",
    "stores": "stores",
    "distributor": "distributors",
    "distributors": "distributors",
    "mn meter": "mn meters",
    "mn meters": "mn meters",
    "mn piece": "mn pieces",
    "mn pieces": "mn pieces",
    "mn customer": "mn customers",
    "mn customers": "mn customers",
    "l sq ft": "l sqft",
    "l sqft": "l sqft",
    "year": "year",
}


def canonical_unit(value):
    text = normalise_text(
        value
    )

    return UNIT_EQUIVALENCE.get(
        text,
        text,
    )


def parse_numeric(value):
    if isinstance(value, bool):
        return None

    if isinstance(value, (int, float)):
        return float(value)

    if not isinstance(value, str):
        return None

    text = value.strip().replace(
        ",",
        "",
    )

    if re.fullmatch(
        r"-?\d+(?:\.\d+)?",
        text,
    ):
        try:
            return float(text)
        except ValueError:
            return None

    return None


def has_source_qualifier(value):
    if not isinstance(value, str):
        return False

    text = value.strip().casefold()

    return (
        text.startswith("~")
        or text.endswith("+")
        or "and counting" in text
    )


def value_equal(reference_value, extracted_value):
    if reference_value is None and extracted_value is None:
        return True

    if (
        has_source_qualifier(reference_value)
        or has_source_qualifier(extracted_value)
    ):
        return (
            normalise_text(reference_value)
            == normalise_text(extracted_value)
        )

    ref_numeric = parse_numeric(
        reference_value
    )

    ext_numeric = parse_numeric(
        extracted_value
    )

    if (
        ref_numeric is not None
        and ext_numeric is not None
    ):
        return math.isclose(
            ref_numeric,
            ext_numeric,
            rel_tol=0.0,
            abs_tol=1e-12,
        )

    return (
        normalise_text(reference_value)
        == normalise_text(extracted_value)
    )


def unit_equal(reference_value, extracted_value):
    return (
        canonical_unit(reference_value)
        == canonical_unit(extracted_value)
    )


def source_location_equal(reference_value, extracted_value):
    return (
        extract_page(reference_value)
        == extract_page(extracted_value)
        and extract_page(reference_value)
        is not None
    )


def description_exact(reference_value, extracted_value):
    return (
        normalise_text(reference_value)
        == normalise_text(extracted_value)
    )


In [ ]:
# ============================================================
# 7. Controlled one-to-one identity alignment
# ============================================================
# HARD BLOCK:
#   Category
#
# IDENTITY EVIDENCE:
#   Topic
#   Reporting Period
#
# EXCLUDED FROM ALIGNMENT:
#   Value
#   Unit
#   Description
#   Source Location


def lexical_similarity(left, right):
    left_norm = normalise_text(left)
    right_norm = normalise_text(right)

    if left_norm == right_norm:
        return 1.0

    if not left_norm or not right_norm:
        return 0.0

    return SequenceMatcher(
        None,
        left_norm,
        right_norm,
    ).ratio()


def topic_similarity(left, right):
    left_norm = normalise_topic(left)
    right_norm = normalise_topic(right)

    if left_norm == right_norm:
        return 1.0

    if not left_norm or not right_norm:
        return 0.0

    return SequenceMatcher(
        None,
        left_norm,
        right_norm,
    ).ratio()


def period_similarity(left, right):
    left_period = canonical_period(left)
    right_period = canonical_period(right)

    if left_period == right_period:
        return 1.0

    if left_period is None or right_period is None:
        return 0.0

    return lexical_similarity(
        left_period,
        right_period,
    )


# ------------------------------------------------------------
# Frozen D11 alignment parameters
# ------------------------------------------------------------

ALIGNMENT_WEIGHTS = {
    "topic": 0.75,
    "reporting_period": 0.25,
}

ALIGNMENT_SCORE_THRESHOLD = 0.60


def alignment_score(
    reference_record,
    extracted_record,
):
    # Category is a hard block.
    if (
        normalise_text(
            reference_record["Category"]
        )
        != normalise_text(
            extracted_record["Category"]
        )
    ):
        return 0.0

    topic_score = topic_similarity(
        reference_record["Topic"],
        extracted_record["Topic"],
    )

    period_score = period_similarity(
        reference_record["Reporting Period"],
        extracted_record["Reporting Period"],
    )

    return (
        ALIGNMENT_WEIGHTS["topic"]
        * topic_score
        +
        ALIGNMENT_WEIGHTS["reporting_period"]
        * period_score
    )


reference_records = (
    reference_df
    .to_dict("records")
)

extraction_records = (
    extracted_df
    .to_dict("records")
)


matches = []

matched_reference_indices = set()
matched_extraction_indices = set()

alignment_diagnostics = []


# ------------------------------------------------------------
# Step 1 — strict one-to-one identity matches
# ------------------------------------------------------------

def strict_identity_key(record):
    return (
        normalise_text(
            record["Category"]
        ),
        normalise_topic(
            record["Topic"]
        ),
        canonical_period(
            record["Reporting Period"]
        ),
    )


reference_identity_map = defaultdict(list)
extraction_identity_map = defaultdict(list)

for index, record in enumerate(
    reference_records
):
    reference_identity_map[
        strict_identity_key(record)
    ].append(index)

for index, record in enumerate(
    extraction_records
):
    extraction_identity_map[
        strict_identity_key(record)
    ].append(index)


reference_duplicate_identity_count = sum(
    len(indices) - 1
    for indices
    in reference_identity_map.values()
    if len(indices) > 1
)

extraction_duplicate_identity_count = sum(
    len(indices) - 1
    for indices
    in extraction_identity_map.values()
    if len(indices) > 1
)


for key in sorted(
    set(reference_identity_map)
    & set(extraction_identity_map)
):

    ref_indices = reference_identity_map[key]
    ext_indices = extraction_identity_map[key]

    if (
        len(ref_indices) == 1
        and len(ext_indices) == 1
    ):
        ref_index = ref_indices[0]
        ext_index = ext_indices[0]

        matches.append({
            "Reference Index":
                ref_index,

            "Extraction Index":
                ext_index,

            "Alignment Rule":
                "Strict Category + Topic + Reporting Period",

            "Alignment Score":
                1.0,

            "Topic Similarity":
                1.0,

            "Reporting Period Similarity":
                1.0,
        })

        matched_reference_indices.add(
            ref_index
        )

        matched_extraction_indices.add(
            ext_index
        )


# ------------------------------------------------------------
# Step 2 — controlled fallback within Category
# ------------------------------------------------------------

categories = sorted(
    {
        normalise_text(
            record["Category"]
        )
        for record
        in reference_records
    }
    |
    {
        normalise_text(
            record["Category"]
        )
        for record
        in extraction_records
    }
)


for category in categories:

    remaining_reference_indices = [
        index
        for index, record
        in enumerate(reference_records)
        if (
            index
            not in matched_reference_indices
            and normalise_text(
                record["Category"]
            ) == category
        )
    ]

    remaining_extraction_indices = [
        index
        for index, record
        in enumerate(extraction_records)
        if (
            index
            not in matched_extraction_indices
            and normalise_text(
                record["Category"]
            ) == category
        )
    ]

    if (
        not remaining_reference_indices
        or not remaining_extraction_indices
    ):
        continue


    score_matrix = np.zeros(
        (
            len(
                remaining_reference_indices
            ),
            len(
                remaining_extraction_indices
            ),
        ),
        dtype=float,
    )


    for row_index, ref_index in enumerate(
        remaining_reference_indices
    ):

        reference_record = (
            reference_records[
                ref_index
            ]
        )

        for column_index, ext_index in enumerate(
            remaining_extraction_indices
        ):

            extracted_record = (
                extraction_records[
                    ext_index
                ]
            )

            score_matrix[
                row_index,
                column_index
            ] = alignment_score(
                reference_record,
                extracted_record,
            )


    row_indices, column_indices = (
        linear_sum_assignment(
            -score_matrix
        )
    )


    for row_index, column_index in zip(
        row_indices,
        column_indices,
    ):

        score = float(
            score_matrix[
                row_index,
                column_index
            ]
        )

        if (
            score
            < ALIGNMENT_SCORE_THRESHOLD
        ):
            continue


        ref_index = (
            remaining_reference_indices[
                row_index
            ]
        )

        ext_index = (
            remaining_extraction_indices[
                column_index
            ]
        )

        reference_record = (
            reference_records[
                ref_index
            ]
        )

        extracted_record = (
            extraction_records[
                ext_index
            ]
        )


        topic_score = topic_similarity(
            reference_record["Topic"],
            extracted_record["Topic"],
        )

        period_score = period_similarity(
            reference_record[
                "Reporting Period"
            ],
            extracted_record[
                "Reporting Period"
            ],
        )


        match = {
            "Reference Index":
                ref_index,

            "Extraction Index":
                ext_index,

            "Alignment Rule":
                "Controlled Category-blocked fallback",

            "Alignment Score":
                score,

            "Topic Similarity":
                topic_score,

            "Reporting Period Similarity":
                period_score,
        }

        matches.append(
            match
        )

        alignment_diagnostics.append(
            match.copy()
        )

        matched_reference_indices.add(
            ref_index
        )

        matched_extraction_indices.add(
            ext_index
        )


# ------------------------------------------------------------
# Final unmatched observations
# ------------------------------------------------------------

missing_reference_indices = sorted(
    set(
        range(
            len(reference_records)
        )
    )
    - matched_reference_indices
)

unsupported_extraction_indices = sorted(
    set(
        range(
            len(extraction_records)
        )
    )
    - matched_extraction_indices
)


strict_alignment_count = sum(
    match[
        "Alignment Rule"
    ].startswith("Strict")
    for match in matches
)

fallback_alignment_count = sum(
    match[
        "Alignment Rule"
    ].startswith("Controlled")
    for match in matches
)


print(
    "Aligned records:",
    len(matches),
)

print(
    "Strict exact alignments:",
    strict_alignment_count,
)

print(
    "Controlled fallback alignments:",
    fallback_alignment_count,
)

print(
    "Missing reference records:",
    len(
        missing_reference_indices
    ),
)

print(
    "Unsupported/unmatched extraction records:",
    len(
        unsupported_extraction_indices
    ),
)

print(
    "Reference duplicate identity count:",
    reference_duplicate_identity_count,
)

print(
    "Extraction duplicate identity count:",
    extraction_duplicate_identity_count,
)

print(
    "Alignment threshold:",
    ALIGNMENT_SCORE_THRESHOLD,
)

print(
    "Alignment weights:",
    ALIGNMENT_WEIGHTS,
)


In [ ]:
# ============================================================
# 8. Field comparison
# ============================================================

comparison_rows = []

for match in matches:

    reference_record = reference_records[
        match["Reference Index"]
    ]

    extracted_record = extraction_records[
        match["Extraction Index"]
    ]

    row = {
        "Reference Index":
            match["Reference Index"],

        "Extraction Index":
            match["Extraction Index"],

        "Alignment Rule":
            match["Alignment Rule"],

        "Alignment Score":
            match.get(
                "Alignment Score",
                1.0,
            ),

        "Topic Similarity":
            match.get(
                "Topic Similarity",
                1.0,
            ),

        "Reporting Period Similarity":
            match.get(
                "Reporting Period Similarity",
                1.0,
            ),
    }

    correctness = {
        "Category":
            normalise_text(
                reference_record["Category"]
            )
            == normalise_text(
                extracted_record["Category"]
            ),

        "Topic":
            normalise_topic(
                reference_record["Topic"]
            )
            == normalise_topic(
                extracted_record["Topic"]
            ),

        "Description":
            description_exact(
                reference_record["Description"],
                extracted_record["Description"],
            ),

        "Value":
            value_equal(
                reference_record["Value"],
                extracted_record["Value"],
            ),

        "Unit":
            unit_equal(
                reference_record["Unit"],
                extracted_record["Unit"],
            ),

        "Reporting Period":
            canonical_period(
                reference_record["Reporting Period"]
            )
            == canonical_period(
                extracted_record["Reporting Period"]
            ),

        "Source Location":
            source_location_equal(
                reference_record["Source Location"],
                extracted_record["Source Location"],
            ),
    }


    all_primary_fields_match = all(
        correctness[field]
        for field in PRIMARY_CORRECTNESS_FIELDS
    )

    identity_fields_match = all(
        correctness[field]
        for field in ALIGNMENT_IDENTITY_FIELDS
    )

    identity_label_difference = (
        all_primary_fields_match
        and not identity_fields_match
    )

    for field in FIELDS:
        row[f"Reference {field}"] = reference_record[field]
        row[f"Extracted {field}"] = extracted_record[field]
        row[f"{field} Correct"] = bool(
            correctness[field]
        )

    row["Fully Correct Primary Record"] = bool(
        all_primary_fields_match
    )

    row["Identity Fields Match"] = bool(
        identity_fields_match
    )

    row["Identity Label Difference"] = bool(
        identity_label_difference
    )

    comparison_rows.append(
        row
    )


comparison_df = pd.DataFrame(
    comparison_rows
)


missing_records_df = (
    reference_df
    .iloc[
        missing_reference_indices
    ]
    .copy()
)

unsupported_records_df = (
    extracted_df
    .iloc[
        unsupported_extraction_indices
    ]
    .copy()
)

discrepant_records_df = (
    comparison_df.loc[
        ~comparison_df[
            "Fully Correct Primary Record"
        ]
    ]
    .copy()
)


print(
    "Fully correct primary records:",
    int(
        comparison_df[
            "Fully Correct Primary Record"
        ].sum()
    ),
)

print(
    "Primary discrepant records:",
    len(
        discrepant_records_df
    ),
)


In [ ]:
# ============================================================
# 9. Metrics
# ============================================================

aligned_records = len(
    comparison_df
)

fully_correct_records = int(
    comparison_df[
        "Fully Correct Primary Record"
    ].sum()
)

discrepant_records = (
    aligned_records
    - fully_correct_records
)

missing_records = len(
    missing_reference_indices
)

unsupported_records = len(
    unsupported_extraction_indices
)

completeness = (
    aligned_records
    / len(reference_df)
    if len(reference_df)
    else 0.0
)

record_precision_exact = (
    fully_correct_records
    / len(extracted_df)
    if len(extracted_df)
    else 0.0
)

record_recall_exact = (
    fully_correct_records
    / len(reference_df)
    if len(reference_df)
    else 0.0
)

record_f1_exact = (
    2
    * record_precision_exact
    * record_recall_exact
    / (
        record_precision_exact
        + record_recall_exact
    )
    if (
        record_precision_exact
        + record_recall_exact
    )
    else 0.0
)


primary_field_accuracy = {}

for field in PRIMARY_CORRECTNESS_FIELDS:

    primary_field_accuracy[field] = (
        float(
            comparison_df[
                f"{field} Correct"
            ].mean()
        )
        if aligned_records
        else 0.0
    )


diagnostic_field_accuracy = {
    "Description exact":
        (
            float(
                comparison_df[
                    "Description Correct"
                ].mean()
            )
            if aligned_records
            else 0.0
        ),
}


field_accuracy = (
    sum(
        primary_field_accuracy.values()
    )
    / len(
        primary_field_accuracy
    )
)


category_metrics = {}

for category, expected_count in (
    EXPECTED_CATEGORY_COUNTS.items()
):

    ref_subset = reference_df[
        reference_df["Category"]
        == category
    ]

    ext_subset = extracted_df[
        extracted_df["Category"]
        == category
    ]

    aligned_subset = comparison_df[
        comparison_df[
            "Reference Category"
        ]
        == category
    ]

    correct_count = int(
        aligned_subset[
            "Fully Correct Primary Record"
        ].sum()
    )

    precision = (
        correct_count
        / len(ext_subset)
        if len(ext_subset)
        else 0.0
    )

    recall = (
        correct_count
        / len(ref_subset)
        if len(ref_subset)
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall
        else 0.0
    )

    category_metrics[category] = {
        "expected_records":
            int(expected_count),

        "extracted_records":
            int(len(ext_subset)),

        "aligned_records":
            int(len(aligned_subset)),

        "fully_correct_records":
            correct_count,

        "discrepant_records":
            int(
                len(aligned_subset)
                - correct_count
            ),

        "completeness":
            float(
                len(aligned_subset)
                / expected_count
                if expected_count
                else 0.0
            ),

        "record_precision_exact":
            float(precision),

        "record_recall_exact":
            float(recall),

        "record_f1_exact":
            float(f1),
    }


print("Reference records:", len(reference_df))
print("Extracted records:", len(extracted_df))
print("Aligned records:", aligned_records)
print("Fully correct records:", fully_correct_records)
print("Discrepant records:", discrepant_records)
print("Missing records:", missing_records)
print("Unsupported/unmatched records:", unsupported_records)
print("Completeness:", round(completeness, 4))
print("Exact F1:", round(record_f1_exact, 4))
print(
    "Overall primary field accuracy:",
    round(
        field_accuracy,
        4,
    ),
)
print("Schema valid:", schema_validity)


In [ ]:
# ============================================================
# 10. Final validation summary
# ============================================================

VALIDATION_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_representation":
        INPUT_REPRESENTATION,

    "alignment_identity_fields":
        ALIGNMENT_IDENTITY_FIELDS,

    "primary_correctness_fields":
        PRIMARY_CORRECTNESS_FIELDS,

    "reference_records":
        int(len(reference_df)),

    "extracted_records":
        int(len(extracted_df)),

    "aligned_records":
        int(aligned_records),

    "fully_correct_records":
        int(fully_correct_records),

    "discrepant_records":
        int(discrepant_records),

    "missing_records":
        int(missing_records),

    "unsupported_extracted_records":
        int(unsupported_records),

    "completeness":
        float(completeness),

    "missing_rate":
        float(
            missing_records
            / len(reference_df)
            if len(reference_df)
            else 0.0
        ),

    "record_precision_exact":
        float(record_precision_exact),

    "record_recall_exact":
        float(record_recall_exact),

    "record_f1_exact":
        float(record_f1_exact),

    "unsupported_rate":
        float(
            unsupported_records
            / len(extracted_df)
            if len(extracted_df)
            else 0.0
        ),

    "discrepancy_rate_among_aligned":
        float(
            discrepant_records
            / aligned_records
            if aligned_records
            else 0.0
        ),

    "field_accuracy":
        float(
            field_accuracy
        ),

    "primary_field_accuracy":
        primary_field_accuracy,

    "diagnostic_field_accuracy":
        diagnostic_field_accuracy,

    "schema_validity":
        bool(schema_validity),

    "schema_diagnostics": {
        "top_level_object_valid":
            bool(
                top_level_object_valid
            ),

        "document_id_correct":
            bool(
                document_id_correct
            ),

        "branch_correct":
            bool(
                branch_correct
            ),

        "records_is_list":
            bool(
                records_is_list
            ),

        "record_schema_valid":
            bool(
                extraction_schema_exact
            ),

        "field_types_valid":
            bool(
                extraction_types_valid
            ),

        "schema_validity":
            bool(
                schema_validity
            ),
    },

    "content_diagnostics": {
        "reference_record_count_valid":
            bool(
                reference_record_count_valid
            ),

        "reference_category_counts_valid":
            bool(
                reference_category_counts_valid
            ),

        "extraction_record_count_valid":
            bool(
                extraction_record_count_valid
            ),

        "extraction_category_counts_valid":
            bool(
                extraction_category_counts_valid
            ),

        "reference_identity_unique":
            bool(
                reference_duplicate_identity_count
                == 0
            ),

        "extraction_duplicate_identity_count":
            int(
                extraction_duplicate_identity_count
            ),

        "strict_alignment_count":
            int(
                strict_alignment_count
            ),

        "fallback_alignment_count":
            int(
                fallback_alignment_count
            ),

        "alignment_score_threshold":
            float(
                ALIGNMENT_SCORE_THRESHOLD
            ),

        "alignment_weights":
            ALIGNMENT_WEIGHTS,

        "remaining_missing_record_count":
            int(
                len(
                    missing_reference_indices
                )
            ),

        "remaining_unsupported_record_count":
            int(
                len(
                    unsupported_extraction_indices
                )
            ),
    },

    "matching_rules": {
        "blocking_field":
            "Category",

        "alignment_identity_fields":
            ALIGNMENT_IDENTITY_FIELDS,

        "fallback_identity_fields": [
            "Topic",
            "Reporting Period",
        ],

        "one_to_one_assignment":
            (
                "Strict identity matching followed by "
                "Category-blocked Hungarian one-to-one assignment "
                "using Topic and Reporting Period only."
            ),

        "matching_score_threshold":
            ALIGNMENT_SCORE_THRESHOLD,

        "matching_score_weights":
            ALIGNMENT_WEIGHTS,

        "value_used_for_alignment":
            False,

        "unit_used_for_alignment":
            False,

        "description_used_for_alignment":
            False,

        "source_location_used_for_alignment":
            False,

        "matching_rules_frozen_across_branches":
            True,
    },

    "comparison_rules": {
        "raw_extraction_modified":
            False,

        "manual_correction_applied":
            False,

        "comparison_normalisation_scope":
            "Comparison copies only",

        "alignment_identity_fields":
            ALIGNMENT_IDENTITY_FIELDS,

        "primary_correctness_fields":
            PRIMARY_CORRECTNESS_FIELDS,

        "description":
            (
                "Normalised exact diagnostic only; excluded from "
                "primary exact-record correctness because the prompt "
                "permits a concise source-grounded description."
            ),

        "numeric_value":
            (
                "Exact represented numeric equality after "
                "deterministic parsing; no tolerance beyond "
                "floating-point representation."
            ),

        "qualified_value":
            (
                "Exact normalised textual agreement with source "
                "qualification preserved; no conversion of ~, +, "
                "or 'and counting' to exact numbers."
            ),

        "text_value":
            (
                "Conservative first-run normalised exact textual "
                "agreement. Any later semantic equivalence must be "
                "source/schema-grounded and then frozen for B/C."
            ),

        "unit":
            (
                "Controlled notation equivalence only; no monetary "
                "rescaling or unit conversion."
            ),

        "source_location":
            (
                "Exact physical PDF page agreement."
            ),

        "d11_equivalence_rules_status":
            (
                "Revised D11 document-level identity alignment is established "
                "in Branch A using Category-blocked, outcome-independent "
                "Topic + Reporting Period evidence. These alignment and "
                "comparison rules must be reused unchanged for Branches B and C."
            ),

        "equivalence_rules_frozen":
            True
    },

    "reference_integrity_confirmation": {
        "reference_semantics_valid":
            bool(
                reference_semantics_valid
            ),

        "checks":
            reference_semantic_checks,

        "reference_modified_by_validation":
            False,
    },

    "category_metrics":
        category_metrics,

    "input_provenance": {
        "reference_file":
            REFERENCE_PATH.name,

        "reference_sha256":
            REFERENCE_SHA256,

        "parsed_extraction_file":
            EXTRACTION_PATH.name,

        "parsed_extraction_sha256":
            EXTRACTION_SHA256,

        "technical_diagnostics_file":
            TECHNICAL_DIAGNOSTICS_PATH.name,

        "technical_diagnostics_sha256":
            TECHNICAL_DIAGNOSTICS_SHA256,

        "experiment_metadata_file":
            METADATA_PATH.name,

        "experiment_metadata_sha256":
            METADATA_SHA256,

        "branch_a_structurally_evaluable":
            bool(
                branch_a_structurally_evaluable
            ),

        "parsed_extraction_hash_matches_metadata":
            bool(
                parsed_extraction_hash_matches_metadata
            ),

        "source_hash_matches_stage_1":
            bool(
                source_hash_matches_stage_1
            ),
    },

    "comparison_rules_frozen_for_branches_B_C":
        True,

}


print(
    json.dumps(
        VALIDATION_SUMMARY,
        ensure_ascii=False,
        indent=2,
    )
)


In [ ]:
# ============================================================
# 11. Export reproducible validation outputs
# ============================================================

SUMMARY_PATH = (
    OUTPUT_DIR
    / "D11_branch_A_validation_summary.json"
)

DETAILED_PATH = (
    OUTPUT_DIR
    / "D11_branch_A_validation_detailed.csv"
)

DISCREPANT_PATH = (
    OUTPUT_DIR
    / "D11_branch_A_discrepant_records.csv"
)

MISSING_PATH = (
    OUTPUT_DIR
    / "D11_branch_A_missing_records.csv"
)

UNSUPPORTED_PATH = (
    OUTPUT_DIR
    / "D11_branch_A_unsupported_records.csv"
)

REFERENCE_SEMANTICS_PATH = (
    OUTPUT_DIR
    / "D11_reference_semantics_confirmation.json"
)

ALIGNMENT_ISSUES_PATH = (
    OUTPUT_DIR
    / "D11_branch_A_alignment_issues.json"
)


SUMMARY_PATH.write_text(
    json.dumps(
        VALIDATION_SUMMARY,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

comparison_df.to_csv(
    DETAILED_PATH,
    index=False,
    encoding="utf-8-sig",
)

discrepant_records_df.to_csv(
    DISCREPANT_PATH,
    index=False,
    encoding="utf-8-sig",
)

missing_records_df.to_csv(
    MISSING_PATH,
    index=False,
    encoding="utf-8-sig",
)

unsupported_records_df.to_csv(
    UNSUPPORTED_PATH,
    index=False,
    encoding="utf-8-sig",
)

REFERENCE_SEMANTICS_PATH.write_text(
    json.dumps(
        {
            "document_id":
                DOCUMENT_ID,

            "reference_semantics_valid":
                reference_semantics_valid,

            "checks":
                reference_semantic_checks,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

ALIGNMENT_ISSUES_PATH.write_text(
    json.dumps(
        {
            "alignment_score_threshold":
                ALIGNMENT_SCORE_THRESHOLD,

            "alignment_weights":
                ALIGNMENT_WEIGHTS,

            "strict_alignment_count":
                strict_alignment_count,

            "fallback_alignment_count":
                fallback_alignment_count,

            "controlled_fallback_alignments":
                alignment_diagnostics,

            "missing_reference_indices":
                missing_reference_indices,

            "unsupported_extraction_indices":
                unsupported_extraction_indices,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

assert reference_semantics_valid
assert branch_a_structurally_evaluable
assert parsed_extraction_hash_matches_metadata
assert source_hash_matches_stage_1
assert schema_validity


required_outputs = [
    SUMMARY_PATH,
    DETAILED_PATH,
    DISCREPANT_PATH,
    MISSING_PATH,
    UNSUPPORTED_PATH,
    REFERENCE_SEMANTICS_PATH,
    ALIGNMENT_ISSUES_PATH,
]

missing_outputs = [
    path.name
    for path in required_outputs
    if not path.exists()
]

if missing_outputs:
    raise AssertionError(
        f"Missing output files: {missing_outputs}"
    )


print("Validation A — D11 completed.")
print(
    "Reference semantics valid:",
    reference_semantics_valid,
)
print(
    "Branch A structurally evaluable:",
    branch_a_structurally_evaluable,
)
print(
    "Aligned / reference:",
    f"{aligned_records}/{len(reference_df)}",
)
print(
    "Fully correct primary records:",
    fully_correct_records,
)
print(
    "Discrepant primary records:",
    discrepant_records,
)
print(
    "Missing records:",
    missing_records,
)
print(
    "Unsupported/unmatched records:",
    unsupported_records,
)
print(
    "Exact F1:",
    round(
        record_f1_exact,
        4,
    ),
)
print(
    "Primary field accuracy:",
    round(
        field_accuracy,
        4,
    ),
)

print(
    "Branch A structurally evaluable:",
    branch_a_structurally_evaluable
)

print(
    "D11 alignment/comparison rules frozen for Branches B/C:",
    True,
)

assert (
    fully_correct_records
    + discrepant_records
    == aligned_records
)

for path in required_outputs:
    print("-", path.name)
